# Phase 3 — Step 1: Cognitive Document Metadata Assignment

## Objective

The Phase 1 ingestion pipeline required a human to assign four **document-level** metadata fields by hand before any chunk was ever produced:

- `clearance_level` (0–3) — the doc's base confidentiality, inherited by prose chunks.
- `allowed_departments` — who may read it (`finance` / `hr` / `engineering` / `legal` / `sales` / `all`).
- `doc_type` — the genre of the document (report, contract, memo, manual…).
- `global_topic` — a one-sentence description used later by the Router-Index.

These values were hardcoded in Phase 2 Step 2 (`07_corpus_augmentation_ingestion.ipynb`) as a `dict` next to each `write_doc(...)` call. That is the *technical debt* this notebook replaces: a production ingestion agent cannot rely on a human filling those fields every time a document is uploaded.

**Scope clarification.** The per-chunk work (PII detection, clearance escalation on isolated fragments) already happens reliably in the existing Custom RBAC regex pipeline. That is kept as-is. This notebook only replaces the *manual document-level tagging* step with an LLM call that runs **once per document**, before the regex chunker takes over.

## Action

For every one of the 23 documents already in the corpus (5 originals + 18 synthetic), we:
1. Read the full raw text.
2. Send it to Llama 3.2 (local, via Ollama) with a strict JSON prompt that encodes the corporate clearance taxonomy, the department vocabulary, the doc-type vocabulary, and a Zero-Trust bias.
3. Compare the 4 predicted fields against the manual ground truth that Phase 2 captured.
4. Measure latency per document.

## What we want to learn

- Can the LLM recover the four fields accurately enough to replace the manual step?
- What is the ingestion-time cost (seconds per document)?
- **In which document types does it fail?** If certain genres are consistently mis-classified, the production system needs a user-override UI, not just a blind LLM call.

In [1]:
# Cell 1 — Imports, paths, Ollama client
import json, os, re, time, statistics
from dataclasses import dataclass
from typing import Optional

import pandas as pd
import requests

BASE         = os.path.dirname(os.path.abspath("__file__"))
RAW_DOCS_DIR = os.path.join(BASE, "..", "..", "data", "raw_docs")
PH1_LOADED   = os.path.join(BASE, "..", "..", "data", "results", "notebook_results", "ph1", "loaded_documents.json")
PH2_CHUNKS   = os.path.join(BASE, "..", "..", "data", "results", "notebook_results", "ph2", "ph2_plus_augmented_chunks.json")
RESULTS_DIR  = os.path.join(BASE, "..", "..", "data", "results", "notebook_results", "ph3")
os.makedirs(RESULTS_DIR, exist_ok=True)

OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL      = "llama3.2"

resp = requests.get("http://localhost:11434/api/tags", timeout=5)
models = [m["name"] for m in resp.json()["models"]]
assert any(MODEL in m for m in models), f"{MODEL} not found in Ollama. Available: {models}"
print(f"Ollama ready. Model in use: {MODEL}")

Ollama ready. Model in use: llama3.2


## Cell 2 — Load the 23 raw documents and the ground-truth metadata

The inputs are the raw files as the ingestion agent would see them:

- **18 synthetic docs** (`.txt` / `.md`): read directly from `raw_docs/`.
- **5 originals** (`.pdf` / `.docx` / `.xlsx` / `.txt`): the pre-extracted text stored in `loaded_documents.json` from Phase 1 Step 1 (so we don't re-run PyPDF / Docx2txt / pandas for Excel here).

The ground truth per document is compiled from `ph2_plus_augmented_chunks.json`: the minimum clearance across all chunks (= the document base, before any per-chunk regex escalation), and the department / doc_type / global_topic which are constant across chunks of the same document.

In [2]:
# Cell 2 — Build the document set with ground-truth metadata

@dataclass
class Doc:
    source:    str
    text:      str
    gt_clearance:   int
    gt_department:  str
    gt_doc_type:    str
    gt_topic:       str

# Per-doc ground truth compiled from Phase-2 chunks metadata
with open(PH2_CHUNKS, "r", encoding="utf-8") as f:
    corpus = json.load(f)

gt: dict[str, dict] = {}
for src, chunks in corpus.items():
    m0 = chunks[0]["metadata"]
    gt[src] = {
        "clearance_level":     min(int(c["metadata"].get("clearance_level", 0)) for c in chunks),
        "allowed_departments": str(m0.get("allowed_departments", "all")).lower(),
        "doc_type":            str(m0.get("doc_type", "unknown")).lower(),
        "global_topic":        str(m0.get("global_topic", m0.get("doc_topic", ""))),
    }

# Pre-extracted text for the 5 originals (PDF/DOCX/XLSX/TXT)
with open(PH1_LOADED, "r", encoding="utf-8") as f:
    loaded = json.load(f)

def read_doc_text(source: str) -> str:
    """Return the full raw text of a document by its source filename."""
    # 1) Try raw_docs/ for plain-text files
    path = os.path.join(RAW_DOCS_DIR, source)
    if os.path.exists(path) and source.lower().endswith((".txt", ".md")):
        with open(path, "r", encoding="utf-8") as fp:
            return fp.read()
    # 2) Binary files: use Phase-1 pre-extracted text
    if source in loaded:
        return "\n".join(part["page_content"] for part in loaded[source])
    raise FileNotFoundError(f"Cannot locate text for {source}")

DOCS: list[Doc] = []
for src in gt:
    text = read_doc_text(src)
    DOCS.append(Doc(
        source=src,
        text=text,
        gt_clearance=gt[src]["clearance_level"],
        gt_department=gt[src]["allowed_departments"],
        gt_doc_type=gt[src]["doc_type"],
        gt_topic=gt[src]["global_topic"],
    ))

# Summary
print(f"Loaded {len(DOCS)} documents ({sum(len(d.text) for d in DOCS):,} total chars)")
print(f"\n  {'Document':<40} {'Chars':>8}  {'cl':>2}  {'dept':<12} {'doc_type':<12} {'topic':<60}")
print("  " + "-" * 138)
for d in DOCS:
    print(f"  {d.source:<40} {len(d.text):>8,}  {d.gt_clearance:>2}  "
          f"{d.gt_department:<12} {d.gt_doc_type:<12} {d.gt_topic[:60]}")

Loaded 23 documents (27,471 total chars)

  Document                                    Chars  cl  dept         doc_type     topic                                                       
  ------------------------------------------------------------------------------------------------------------------------------------------
  Witty-QuickGuide-EN.pdf                     2,249   0  all          manual       Witty Timer hardware quick start guide
  Witty-Financial-Report-2025.pdf             2,458   3  finance      report       Witty product line Q3 2025 financial performance
  distribution-contract-2026.docx             1,059   2  legal        contract     Exclusive distribution contract for Witty Timer in Spain
  server_logs_witty_backend.txt                 653   2  engineering  log          Witty Manager API backend server logs
  clients-and-billings.xlsx                     538   3  sales        spreadsheet  Client database with billing and support tier info
  fin_q1_2026_revenue.tx

## Cell 3 — Strategy: the LLM classifier (full-document, single call)

The prompt was tuned with the following design choices:

1. **Controlled vocabularies** — the prompt enumerates the exact allowed values for `allowed_departments` and `doc_type`. This eliminates the "creative paraphrase" failure mode (the model returning `"HR"` or `"human-resources"` instead of `"hr"`). Values are case-normalised on parse.
2. **Clearance taxonomy with worked examples** — each of the 4 levels has 2–3 examples, including well-known traps (a password *policy* is `1`, a password *value* is `3`).
3. **Zero-Trust bias** — the prompt instructs the model to round UP when uncertain. This matches the Phase-2 design where False Positives (over-classification) cause UX friction but never data leaks, while False Negatives are the dangerous failure mode.
4. **Few-shot anchors** — three worked examples (one obvious `cl=3`, one ambiguous `cl=1`, one public `cl=0`) calibrate the model's interpretation of the taxonomy.
5. **Strict JSON output** — Ollama's `format: json` parameter guarantees well-formed JSON; we additionally normalise and clamp values after parsing.
6. **Fail-safe on parse error** — if JSON parsing fails even after retry, the classifier returns `cl=3` with a flag, so a malformed LLM response never under-classifies a document.

In [3]:
# Cell 3 — LLM classifier implementation

ALLOWED_DEPTS = ["finance", "hr", "engineering", "legal", "sales", "all"]
ALLOWED_DOC_TYPES = [
    "report", "contract", "log", "memo", "minutes", "directory",
    "guide", "manual", "policy", "spreadsheet", "changelog", "unknown",
]

SYSTEM_PROMPT = """You are the Document Ingestion Classifier for an enterprise on-premise RAG system with Role-Based Access Control.

You read ONE full document and output FOUR metadata fields in a single JSON object. These fields travel with every chunk produced from the document, so they must reflect the document as a whole.

===========================================================
FIELD 1 — clearance_level (int 0..3)
===========================================================
  0 = PUBLIC        Product manuals, marketing materials, public FAQs, user guides of shipped products.
  1 = INTERNAL      Company-wide policies, hygiene rules, onboarding, non-sensitive config, internal memos without PII.
                    A password POLICY is 1, NOT 3 (the rule is not a secret).
  2 = CONFIDENTIAL  Client contracts, SLAs, incident reports, server logs, department-scoped procedural knowledge,
                    competitive intelligence, moderate financial detail.
  3 = STRICT        Raw credentials (password values, tokens, API keys), IBAN/SWIFT records, DNI/national-ID tables,
                    salary bands, financial reports, board minutes, M&A or layoff plans, strategic memos.

ZERO-TRUST BIAS: when uncertain between two levels, pick the HIGHER one. Under-classifying is a data leak.

===========================================================
FIELD 2 — allowed_departments
===========================================================
Pick ONE of exactly: finance | hr | engineering | legal | sales | all
  "all" is ONLY for company-wide content (public manuals, company-wide policies, onboarding for everyone).
  Every confidential document belongs to one specific department.

===========================================================
FIELD 3 — doc_type
===========================================================
Pick ONE of exactly: report | contract | log | memo | minutes | directory | guide | manual | policy | spreadsheet | changelog
  - report: analytical or quarterly document with findings and numbers
  - contract / SLA: legal agreement between parties
  - log: time-stamped system output
  - memo: short internal announcement or update
  - minutes: record of a meeting
  - directory: structured listing of people/clients
  - guide / manual: how-to documentation (guide = internal, manual = user-facing)
  - policy: rules the company enforces
  - spreadsheet: tabular data export
  - changelog: version release notes

===========================================================
FIELD 4 — global_topic
===========================================================
One sentence, 8–15 words, describing what the document is about. No PII. No proper nouns of individuals.

===========================================================
OUTPUT FORMAT — single JSON object, no prose, no markdown:
===========================================================
{
  "clearance_level": <int 0..3>,
  "allowed_departments": "<one of the 6 allowed>",
  "doc_type": "<one of the 11 allowed>",
  "global_topic": "<8-15 words>",
  "reasoning": "<max 25 words explaining the key signals>"
}

FEW-SHOT EXAMPLES

Example A — Board minutes document:
Input: "CONFIDENTIAL - BOARD MEETING MINUTES. Date: March 28, 2026. Attendees: CEO, CFO, CTO. 1. Financial Review: Q1 revenue €5.1M, 62% gross margin. Board approved 2027 R&D budget €2.8M. ..."
Response: {"clearance_level":3,"allowed_departments":"finance","doc_type":"minutes","global_topic":"Board meeting minutes covering quarterly financials and 2027 budget approvals","reasoning":"Board-level decisions with explicit financial figures, strict."}

Example B — Product user manual:
Input: "Witty Timer Specs. Physical: 145 x 95 x 35 mm, 280g. Operating temperature -10 to +50°C. IP65. Wireless 2.4 GHz proprietary ..."
Response: {"clearance_level":0,"allowed_departments":"all","doc_type":"manual","global_topic":"Technical specifications and hardware details of the Witty Timer product","reasoning":"Public product specs, safe to share externally."}

Example C — Internal password policy:
Input: "Corporate Password Policy v2.1. Minimum 12 characters, rotation every 90 days, no reuse across systems."
Response: {"clearance_level":1,"allowed_departments":"all","doc_type":"policy","global_topic":"Company-wide rules for password complexity and rotation","reasoning":"Policy rules, not secret values."}
"""

def classify_document(text: str, retries: int = 1) -> dict:
    """Classify a whole document in one LLM call. Returns the 4 metadata fields + latency."""
    prompt = f"{SYSTEM_PROMPT}\n\nNOW CLASSIFY THIS DOCUMENT:\n---\n{text}\n---\nResponse:"
    t0 = time.perf_counter()
    last_err: Optional[str] = None
    for attempt in range(retries + 1):
        try:
            resp = requests.post(OLLAMA_URL, json={
                "model": MODEL,
                "prompt": prompt,
                "stream": False,
                "format": "json",
                "options": {"temperature": 0.0, "num_predict": 260, "num_ctx": 8192},
            }, timeout=180)
            raw = resp.json().get("response", "").strip()
            raw = re.sub(r"^```(?:json)?|```$", "", raw, flags=re.IGNORECASE).strip()
            parsed = json.loads(raw)
            cl = max(0, min(3, int(parsed.get("clearance_level", 0))))
            dept = str(parsed.get("allowed_departments", "all")).lower().strip()
            if dept not in ALLOWED_DEPTS:
                dept = "all"
            dt = str(parsed.get("doc_type", "unknown")).lower().strip()
            if dt not in ALLOWED_DOC_TYPES:
                dt = "unknown"
            return {
                "clearance_level":     cl,
                "allowed_departments": dept,
                "doc_type":            dt,
                "global_topic":        str(parsed.get("global_topic", ""))[:220],
                "reasoning":           str(parsed.get("reasoning", ""))[:240],
                "latency_s":           round(time.perf_counter() - t0, 2),
                "parse_retries":       attempt,
                "raw":                 raw,
            }
        except (json.JSONDecodeError, ValueError, KeyError, TypeError) as e:
            last_err = f"{type(e).__name__}: {e}"
            continue
    # Total parse failure → fail-safe, assume STRICT
    return {
        "clearance_level":     3,
        "allowed_departments": "all",
        "doc_type":            "unknown",
        "global_topic":        "",
        "reasoning":           f"Fail-safe: JSON parse failed ({last_err}). Assumed strict.",
        "latency_s":           round(time.perf_counter() - t0, 2),
        "parse_retries":       retries + 1,
        "raw":                 "",
    }

# Smoke test on the first document
smoke = classify_document(DOCS[0].text)
print(f"Smoke test on {DOCS[0].source}:")
print(f"  gt  -> cl={DOCS[0].gt_clearance} dept={DOCS[0].gt_department} type={DOCS[0].gt_doc_type}")
print(f"  llm -> cl={smoke['clearance_level']} dept={smoke['allowed_departments']} type={smoke['doc_type']} "
      f"({smoke['latency_s']}s)")
print(f"  topic  (gt)  : {DOCS[0].gt_topic}")
print(f"  topic  (llm) : {smoke['global_topic']}")

Smoke test on Witty-QuickGuide-EN.pdf:
  gt  -> cl=0 dept=all type=manual
  llm -> cl=0 dept=all type=guide (19.73s)
  topic  (gt)  : Witty Timer hardware quick start guide
  topic  (llm) : Quick guide for using the Witty Manager software and hardware, including user manual and installation instructions.


## Cell 4 — Run the classifier over all 23 documents

In [4]:
# Cell 4 — Execute LLM classifier on every document

records: list[dict] = []
t_start = time.time()

print(f"Classifying {len(DOCS)} documents with {MODEL}...\n")
print(f"  {'#':<3} {'Document':<40} {'gt cl':>5} {'llm cl':>7} {'gt dept':<13} {'llm dept':<13} {'gt type':<12} {'llm type':<12} {'lat':>6}")
print("  " + "-" * 124)

for i, d in enumerate(DOCS, 1):
    out = classify_document(d.text)

    rec = {
        "source":            d.source,
        "chars":             len(d.text),
        "gt_clearance":      d.gt_clearance,
        "gt_department":     d.gt_department,
        "gt_doc_type":       d.gt_doc_type,
        "gt_topic":          d.gt_topic,
        "llm_clearance":     out["clearance_level"],
        "llm_department":    out["allowed_departments"],
        "llm_doc_type":      out["doc_type"],
        "llm_topic":         out["global_topic"],
        "llm_reasoning":     out["reasoning"],
        "latency_s":         out["latency_s"],
        "parse_retries":     out["parse_retries"],
    }
    # Per-field correctness
    rec["clearance_exact"]   = rec["llm_clearance"]   == rec["gt_clearance"]
    rec["clearance_fn"]      = rec["llm_clearance"]    < rec["gt_clearance"]    # leak
    rec["clearance_fp"]      = rec["llm_clearance"]    > rec["gt_clearance"]    # friction
    rec["department_exact"]  = rec["llm_department"]  == rec["gt_department"]
    rec["doc_type_exact"]    = rec["llm_doc_type"]    == rec["gt_doc_type"]

    records.append(rec)

    mark = "OK" if rec["clearance_exact"] else ("FN" if rec["clearance_fn"] else "FP")
    print(f"  {i:<3} {d.source[:40]:<40} {d.gt_clearance:>5} {rec['llm_clearance']:>7} "
          f"{d.gt_department:<13} {rec['llm_department']:<13} "
          f"{d.gt_doc_type:<12} {rec['llm_doc_type']:<12} {rec['latency_s']:>5.1f}s  [{mark}]")

df = pd.DataFrame(records)
print(f"\nTotal wall-clock: {time.time() - t_start:.0f}s for {len(DOCS)} documents.")

Classifying 23 documents with llama3.2...

  #   Document                                 gt cl  llm cl gt dept       llm dept      gt type      llm type        lat
  ----------------------------------------------------------------------------------------------------------------------------


  1   Witty-QuickGuide-EN.pdf                      0       0 all           all           manual       guide          6.5s  [OK]


  2   Witty-Financial-Report-2025.pdf              3       3 finance       finance       report       report         7.7s  [OK]


  3   distribution-contract-2026.docx              2       3 legal         legal         contract     contract       7.8s  [FP]


  4   server_logs_witty_backend.txt                2       2 engineering   all           log          log            7.2s  [OK]


...

  19  product_photocell_alignment.md               0       0 all           all           manual       guide          6.6s  [OK]


  20  product_release_notes_v4.txt                 0       0 all           all           changelog    unknown        6.8s  [OK]


  21  memo_q2_sales_targets.txt                    1       1 sales         sales         memo         memo           6.4s  [OK]


  22  memo_it_security_reminder.txt                1       1 all           all           memo         memo           6.4s  [OK]


  23  memo_board_meeting_minutes.txt               3       3 finance       finance       minutes      minutes        7.0s  [OK]

Total wall-clock: 157s for 23 documents.


## Cell 5 — Accuracy per field + latency

Four separate numbers, because the fields play different roles downstream:

- **`clearance_level`** — the security-critical one. Split into exact / False Negative (leak) / False Positive (UX friction).
- **`allowed_departments`** — exact match against controlled vocabulary.
- **`doc_type`** — exact match against controlled vocabulary.
- **`global_topic`** — free text, not scored automatically; printed side-by-side for qualitative review.

In [5]:
# Cell 5 — Per-field accuracy + latency

N = len(df)
pct = lambda n: f"{n / N * 100:5.1f}%"

print("=" * 78)
print("  PER-FIELD ACCURACY  (N = {} documents)".format(N))
print("=" * 78)
print(f"\n  clearance_level       exact: {pct(df['clearance_exact'].sum())}  "
      f"FN (leak): {pct(df['clearance_fn'].sum())}  "
      f"FP (friction): {pct(df['clearance_fp'].sum())}")
print(f"  allowed_departments   exact: {pct(df['department_exact'].sum())}")
print(f"  doc_type              exact: {pct(df['doc_type_exact'].sum())}")
print(f"  global_topic          (see qualitative comparison below)")

# Latency stats
lat = df["latency_s"].tolist()
print("\n" + "=" * 78)
print("  LATENCY (seconds per document)")
print("=" * 78)
print(f"  mean:   {statistics.mean(lat):.2f}s")
print(f"  median: {statistics.median(lat):.2f}s")
print(f"  min:    {min(lat):.2f}s     max: {max(lat):.2f}s")
print(f"  total:  {sum(lat):.0f}s for {N} docs (this is a one-time ingestion cost per document).")

# Per-clearance breakdown (how well the LLM handles each severity)
print("\n" + "=" * 78)
print("  clearance_level — per ground-truth severity")
print("=" * 78)
print(f"  {'gt':>3}  {'n':>3}  {'exact':>7}  {'FN':>4}  {'FP':>4}")
for gt_cl in [0, 1, 2, 3]:
    sub = df[df["gt_clearance"] == gt_cl]
    if len(sub) == 0:
        continue
    n = len(sub)
    print(f"  {gt_cl:>3}  {n:>3}  {sub['clearance_exact'].sum() / n * 100:>6.0f}%  "
          f"{sub['clearance_fn'].sum():>4}  {sub['clearance_fp'].sum():>4}")

  PER-FIELD ACCURACY  (N = 23 documents)

  clearance_level       exact:  65.2%  FN (leak):  17.4%  FP (friction):  17.4%
  allowed_departments   exact:  73.9%
  doc_type              exact:  65.2%
  global_topic          (see qualitative comparison below)

  LATENCY (seconds per document)
  mean:   6.83s
  median: 6.83s
  min:    5.75s     max: 7.82s
  total:  157s for 23 docs (this is a one-time ingestion cost per document).

  clearance_level — per ground-truth severity
   gt    n    exact    FN    FP
    0    5      80%     0     1
    1    4      50%     0     2
    2    7      71%     1     1
    3    7      57%     3     0


## Cell 6 — Failure analysis

For every document where any of the three scored fields diverges from ground truth, we print the diff and the LLM's reasoning. This is where the production recommendation is decided:

- If failures cluster on a specific *genre* (e.g. always the same doc type), the fallback is a **user-override UI restricted to that genre** — much cheaper than a full manual step.
- If failures are scattered and unpredictable, we need a **mandatory confirmation step** where the user approves the LLM output before ingestion completes.

In [6]:
# Cell 6 — Detailed failures + qualitative topic comparison

failures = df[~(df["clearance_exact"] & df["department_exact"] & df["doc_type_exact"])]
print("=" * 78)
print(f"  DOCUMENTS WITH AT LEAST ONE FIELD MISMATCH: {len(failures)} / {N}")
print("=" * 78)
for _, r in failures.iterrows():
    diffs = []
    if not r["clearance_exact"]:
        tag = "FN" if r["clearance_fn"] else "FP"
        diffs.append(f"clearance gt={r['gt_clearance']} -> llm={r['llm_clearance']} [{tag}]")
    if not r["department_exact"]:
        diffs.append(f"dept gt={r['gt_department']} -> llm={r['llm_department']}")
    if not r["doc_type_exact"]:
        diffs.append(f"doc_type gt={r['gt_doc_type']} -> llm={r['llm_doc_type']}")
    print(f"\n  [{r['source']}]")
    for d in diffs:
        print(f"      · {d}")
    print(f"      reasoning: {r['llm_reasoning']}")

# Topic comparison (qualitative, 5 random samples)
print("\n" + "=" * 78)
print("  GLOBAL_TOPIC — side-by-side for 5 documents")
print("=" * 78)
sample = df.sample(min(5, len(df)), random_state=0).sort_values("source")
for _, r in sample.iterrows():
    print(f"\n  [{r['source']}]")
    print(f"    gt  : {r['gt_topic']}")
    print(f"    llm : {r['llm_topic']}")

  DOCUMENTS WITH AT LEAST ONE FIELD MISMATCH: 16 / 23

  [Witty-QuickGuide-EN.pdf]
      · doc_type gt=manual -> llm=guide
      reasoning: Public product specs, safe to share externally. No sensitive information or confidential content.

  [distribution-contract-2026.docx]
      · clearance gt=2 -> llm=3 [FP]
      reasoning: Contract with explicit financial details (IBAN/SWIFT record), strict confidentiality clauses, and legal department involvement.

  [server_logs_witty_backend.txt]
      · dept gt=engineering -> llm=all
      reasoning: Log entries with system alerts, error messages, and recovery actions indicate a critical incident requiring immediate attention

  [clients-and-billings.xlsx]
      · clearance gt=3 -> llm=2 [FN]
      · dept gt=sales -> llm=all
      · doc_type gt=spreadsheet -> llm=directory
      reasoning: Client directory with names, email addresses, and facturación (billing) amounts, indicating a general overview of client data

  [fin_q1_2026_revenue.txt]
  

## Cell 7 — Export

In [7]:
# Cell 7 — Persist results

out = {
    "step": "Phase 3 - Step 1: Cognitive Document Metadata Assignment",
    "model": MODEL,
    "n_documents": N,
    "per_field_accuracy": {
        "clearance_level": {
            "exact_pct": round(df["clearance_exact"].sum() / N * 100, 1),
            "fn_pct":    round(df["clearance_fn"].sum()    / N * 100, 1),
            "fp_pct":    round(df["clearance_fp"].sum()    / N * 100, 1),
        },
        "allowed_departments": {
            "exact_pct": round(df["department_exact"].sum() / N * 100, 1),
        },
        "doc_type": {
            "exact_pct": round(df["doc_type_exact"].sum() / N * 100, 1),
        },
    },
    "latency_s": {
        "mean":   round(statistics.mean(lat), 2),
        "median": round(statistics.median(lat), 2),
        "min":    round(min(lat), 2),
        "max":    round(max(lat), 2),
        "total":  round(sum(lat), 1),
    },
    "records": records,
}

json_path = os.path.join(RESULTS_DIR, "ph3_step1_metadata_classifier.json")
csv_path  = os.path.join(RESULTS_DIR, "ph3_step1_metadata_classifier.csv")
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(out, f, indent=2, ensure_ascii=False, default=str)
df.to_csv(csv_path, index=False)
print(f"Exported: {json_path}")
print(f"Exported: {csv_path}")

print("\n" + "=" * 72)
print("  STEP 1 SUMMARY")
print("=" * 72)
print(f"  Documents classified : {N}")
print(f"  clearance_level exact: {df['clearance_exact'].sum() / N * 100:.1f}%  "
      f"(FN {df['clearance_fn'].sum()}/{N}, FP {df['clearance_fp'].sum()}/{N})")
print(f"  department     exact : {df['department_exact'].sum() / N * 100:.1f}%")
print(f"  doc_type       exact : {df['doc_type_exact'].sum() / N * 100:.1f}%")
print(f"  Mean latency         : {statistics.mean(lat):.2f}s per doc")
print("\n  Awaiting approval before Step 2 (Ingestion Agent & Router-Index).")

Exported: D:\URV\TFG\ai-rag-context-auth-system\strategy-analysis\notebooks\ph3-cognitive-pipeline\..\..\data\results\notebook_results\ph3\ph3_step1_metadata_classifier.json
Exported: D:\URV\TFG\ai-rag-context-auth-system\strategy-analysis\notebooks\ph3-cognitive-pipeline\..\..\data\results\notebook_results\ph3\ph3_step1_metadata_classifier.csv

  STEP 1 SUMMARY
  Documents classified : 23
  clearance_level exact: 65.2%  (FN 4/23, FP 4/23)
  department     exact : 73.9%
  doc_type       exact : 65.2%
  Mean latency         : 6.83s per doc

  Awaiting approval before Step 2 (Ingestion Agent & Router-Index).
